In [0]:
import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from mlflow.models.signature import infer_signature

#Set experiment

In [0]:
mlflow.set_experiment("/Users/approjects8473@gmail.com/sales_forecasting_v3")
print("MLflow experiment set up successfully")

#Load Gold Tables

In [0]:
fact_sales = spark.table("workspace.gold.fact_sales").toPandas()
dim_customers = spark.table("workspace.gold.dim_customers").toPandas()
dim_products = spark.table("workspace.gold.dim_products").toPandas()

print(f"fact_sales: {len(fact_sales)} rows")
print(f"dim_customers: {len(dim_customers)} rows")
print(f"dim_products: {len(dim_products)} rows")

#Join fact_sales with dimensions

In [0]:
df = fact_sales.merge(dim_customers, on="customer_key", how="left")
df = df.merge(dim_products, on="product_key", how="left")

#Extract time features from order_date

In [0]:
df["order_date"] = pd.to_datetime(df["order_date"])
df["order_month"] = df["order_date"].dt.month
df["order_year"] = df["order_date"].dt.year
df["order_dayofweek"] = df["order_date"].dt.dayofweek

#Select features and target

In [0]:
features = [
    "quantity",
    "price",
    "order_month",
    "order_year",
    "order_dayofweek",
]

target = "sales_amount"

#Drop rows with null values 

In [0]:
df_model = df[features + [target]].dropna()

In [0]:
print(f"Training dataset: {len(df_model)} rows, {len(features)} features")
print(f"Features: {features}")
print(f"Target: {target}")
print(df_model.head())

#Train model with MLflow tracking 

In [0]:
X = df_model[features]
y = df_model[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

with mlflow.start_run(run_name="random_forest_v1"):
    # Parameters
    n_estimators = 100
    max_depth = 10
    
    mlflow.log_param("n_estimators", n_estimators)
    mlflow.log_param("max_depth", max_depth)
    mlflow.log_param("features", str(features))
    mlflow.log_param("train_size", len(X_train))
    mlflow.log_param("test_size", len(X_test))
    
    # Train
    model = RandomForestRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )
    model.fit(X_train, y_train)
    
    # Evaluate
    predictions = model.predict(X_test)
    mae = mean_absolute_error(y_test, predictions)
    r2 = r2_score(y_test, predictions)

    mlflow.log_metric("mae", mae)
    mlflow.log_metric("r2", r2)

    signature = infer_signature(X_train, model.predict(X_train))
    
    # Log model
    mlflow.sklearn.log_model(
        model, 
        artifact_path="sales_forecast_model",
        registered_model_name="sales_forecasting_v3",
        signature=signature
    )

    print(f"Training complete!")
    print(f"MAE:  {mae:.2f}")
    print(f"R2:   {r2:.4f}")
    print(f"Model registered as: sales_forecasting_v3")

In [0]:
import requests
import json

#Get the token

In [0]:
token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
host = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().get()

#Call the endpoint

In [0]:
url = f"{host}/serving-endpoints/sales-forecast-endpoint/invocations"

payload = {
    "dataframe_records": [
        {
            "quantity": 2,
            "price": 1500.0,
            "order_month": 6,
            "order_year": 2026,
            "order_dayofweek": 2
        }
    ]
}

response = requests.post(
    url,
    headers={"Authorization": f"Bearer {token}"},
    json=payload
)

print(f"Status code: {response.status_code}")
print(f"Prediction: {response.json()}")